# Week 6: Query Optimization & Data Quality

**Name:** Hashim Nadeem  
**Roll No:** SU92-BDSFM-F25-001  
**Section:** BS DS-7a  
**Implementation:** Python, pandas and SQLite.

This notebook contains the complete dual-route load and T1–T7 answers. Run all cells from this folder. The script version provides the same reproducible pipeline.

## Source-contract decisions

1. **Physical records:** the supplied road CSV has 708 data records plus one empty record and one TOTAL footer. RawLanding retains all **1,286** physical source records; StructuralAudit retains the two non-consignment records; staging has **1,284** data rows. The brief contradicts itself by including the footer/blank in its stated 1,284. Both identities are explicitly checked.
2. **Weight syntax:** to reproduce the stated nine text-weight rejects, the source contract accepts ordinary decimal numeric syntax. The road token `1.2e4` is valid scientific notation mathematically, but is a text exception under this exercise contract; it is quarantined for source confirmation, not guessed or silently discarded. Accepting scientific notation would add one row and 12,000 kg, contrary to the supplied controls.
3. **Types before profiling:** XLSX is read with `dtype=object`; otherwise pandas converts rail consignment strings into integers and erases two trailing-space variants. CSV fields retain their original text. Distinct raw counts exclude blanks and preserve source-specific representations.
4. **Date period:** 90 days means Q1 2026 (1 January–31 March), not March alone. No holiday calendar was supplied; `IsHoliday` is explicitly Unknown.
5. **Fact contract:** eight identifiers = ShipmentID plus seven dimension FKs; three facts = WeightKG, FreightCostPKR and ShipmentCount. Staging is intermediate data in pandas and is exported only as audit evidence, not loaded into the analytical database.

## Setup

In [1]:
"""PakFreight Week 6. Hashim Nadeem, SU92-BDSFM-F25-001, BS DS-7a.
Run from this folder. Python/pandas implementation; source inputs remain unchanged.
"""
from pathlib import Path
import csv, json, re, sqlite3
from datetime import datetime, timezone
import numpy as np
import pandas as pd
try:
    BASE = Path(__file__).resolve().parent
except NameError:
    BASE = Path.cwd()
DATA = BASE / 'data'
OUT = BASE / 'outputs'
OUT.mkdir(exist_ok=True)
CONTRACT = ['ShipmentID','ShipDate','Origin','Destination','WeightKG','FreightCostPKR','StatusRaw','CustomerID']
RUN_ID = 'PakFreight-March-2026-v1'
AUDIT_TIME = datetime.now(timezone.utc).isoformat()
pd.set_option('display.max_columns', 15)

## Step 01 - Immutable extraction with physical row lineage

In [2]:
# CSV reader keeps the empty record. Rail header is physical row 3.
SPECS = [
 ('marine_march.csv','Marine',['Shipment_ID','Ship_Date','Origin_Port','Dest_Port','Weight_KG','Cost_PKR','Status','Customer_Ref']),
 ('rail_march.xlsx','Rail',['consignment_no','date','from_station','to_station','net_tons','freight_charges','consignment_status','customer_code']),
 ('road_march.csv','Road',['trip_id','trip_date','from_city','to_city','weight_kg','cost_pkr','status','customer_id'])]
frames=[]
for filename, mode, source_cols in SPECS:
    path=DATA/'march'/filename
    if path.suffix=='.csv':
        with path.open(newline='', encoding='utf-8-sig') as f:
            reader=csv.reader(f); header=next(reader); records=[]
            for row in reader:
                line=reader.line_num
                row=(row+['']*len(header))[:len(header)]
                records.append(dict(zip(header,row), SourceRow=line))
        df=pd.DataFrame(records)
    else:
        df=pd.read_excel(path,header=2,keep_default_na=False,dtype=object)
        df['SourceRow']=np.arange(4,len(df)+4)
    raw_cols=list(source_cols)
    df['RawRecord']=df[raw_cols].apply(lambda r:json.dumps(r.to_dict(),default=str,ensure_ascii=False),axis=1)
    df=df.rename(columns=dict(zip(source_cols,CONTRACT)))
    df['SourceFile']=filename; df['SourceMode']=mode
    frames.append(df[CONTRACT+['SourceFile','SourceRow','SourceMode','RawRecord']])
landing=pd.concat(frames,ignore_index=True)
landing.to_csv(OUT/'RawLanding.csv',index=False)
print('Physical landing records:',len(landing))

Physical landing records: 1286


## Step 02 - Shape contract; retain structural exceptions separately

In [3]:
# Brief inconsistency: 261 + 315 + 708 = 1284 data rows, but road has
# two additional structural records. Nothing is silently lost: 1286 = 1284 + 2.
all_blank=landing[CONTRACT].apply(lambda c:c.astype(str).str.strip().eq('')).all(axis=1)
footer=landing.ShipmentID.astype(str).str.strip().str.upper().eq('TOTAL')
structural=landing[all_blank|footer].copy()
structural['Reason']=np.where(footer[all_blank|footer],'TOTAL footer; not a consignment','Blank structural record')
structural.to_csv(OUT/'StructuralAudit.csv',index=False)
staging=landing[~(all_blank|footer)].copy().reset_index(drop=True)
staging.to_csv(OUT/'StagingRaw.csv',index=False)
assert len(landing)==len(staging)+len(structural)
print('Staging data rows:',len(staging),'Structural records retained:',len(structural))

Staging data rows: 1284 Structural records retained: 2


## Step 03 - Profile untouched contract values

In [4]:
aliases=pd.read_csv(DATA/'alias_lookup.csv',keep_default_na=False)
registry=pd.read_csv(DATA/'customers_master.csv',keep_default_na=False)
dim_city=pd.read_csv(DATA/'dims/DimCity.csv',keep_default_na=False)
dim_route=pd.read_csv(DATA/'dims/DimRoute.csv',keep_default_na=False)
dim_mode=pd.read_csv(DATA/'dims/DimMode.csv',keep_default_na=False)
dim_status=pd.read_csv(DATA/'dims/DimStatus.csv',keep_default_na=False)
stale_date=pd.read_csv(DATA/'dims/DimDate.csv',keep_default_na=False)
def trim(v): return str(v).strip()
def norm(v): return ' '.join(trim(v).upper().split())
def blank(v): return trim(v)==''
def decimal_number(v):
    # Source contract accepts ordinary signed decimal numbers only.
    # Scientific notation is not issued by these source systems; 1.2e4 is quarantined.
    if isinstance(v,(int,float,np.number)): return float(v) if np.isfinite(v) else np.nan
    t=trim(v)
    return float(t) if re.fullmatch(r'[+-]?(?:\d+(?:\.\d*)?|\.\d+)',t) else np.nan

def parse_date(row):
    v=row.ShipDate
    if isinstance(v,(datetime,pd.Timestamp)): return pd.Timestamp(v).normalize()
    fmt='%d/%m/%Y' if row.SourceMode=='Marine' else '%Y-%m-%d'
    try:return pd.Timestamp(datetime.strptime(trim(v),fmt))
    except (ValueError,TypeError):return pd.NaT

id_norm=staging.ShipmentID.map(norm)
parsed_dates=staging.apply(parse_date,axis=1)
weights=staging.WeightKG.map(decimal_number)
costs=staging.FreightCostPKR.map(decimal_number)
alias_key=dict(zip(aliases.RawSpelling.map(norm),aliases.CityKey))
status_key=dict(zip(dim_status.StatusName.map(norm),dim_status.StatusKey))
origin=staging.Origin.map(norm).map(alias_key)
destination=staging.Destination.map(norm).map(alias_key)
status=staging.StatusRaw.map(norm).map(status_key)
failures=pd.DataFrame({
 'ShipmentID':id_norm.eq('') | (id_norm.duplicated() & id_norm.ne('')),
 'ShipDate':parsed_dates.isna() | ~parsed_dates.between('2026-03-01','2026-03-31'),
 'Origin':origin.isna(),'Destination':destination.isna(),
 'WeightKG':weights.isna() | weights.le(0),
 'FreightCostPKR':costs.isna(),'StatusRaw':status.isna(),
 'CustomerID':~staging.CustomerID.map(trim).isin(registry.CustomerID)})
city_names=dict(zip(dim_city.CityKey,dim_city.CityName))
text_transforms={
 'ShipmentID':((staging.ShipmentID.astype(str)!=id_norm)&id_norm.ne('')).sum(),
 'Origin':(origin.notna() & staging.Origin.astype(str).ne(origin.map(city_names))).sum(),
 'Destination':(destination.notna() & staging.Destination.astype(str).ne(destination.map(city_names))).sum(),
 'StatusRaw':(status.notna() & staging.StatusRaw.astype(str).ne(status.map(dict(zip(dim_status.StatusKey,dim_status.StatusName))))).sum()}
profile=pd.DataFrame([{'Column':c,'Blanks':int(staging[c].map(blank).sum()),
 'DistinctRaw':int(staging.loc[~staging[c].map(blank),c].map(str).nunique()),
 'RuleFailures':int(failures[c].sum()),'NeedsTextTransform':text_transforms.get(c,'—')} for c in CONTRACT])
profile.to_csv(OUT/'T2_Profile.csv',index=False)
assert profile.RuleFailures.sum()==105
print(profile.to_string(index=False))

        Column  Blanks  DistinctRaw  RuleFailures NeedsTextTransform
    ShipmentID       4         1232            64                712
      ShipDate       0           98             6                  —
        Origin       0           18            14                570
   Destination       0           19             0                189
      WeightKG       0           78            14                  —
FreightCostPKR       0          378             0                  —
     StatusRaw       0           25             7                980
    CustomerID       0           46             0                  —


## Step 04 - Reject only unrecoverable consignment fields

In [5]:
# Retain all applicable reasons, not merely the first failed validation.
reasons=[]
for idx,row in staging.iterrows():
    r=[]
    if id_norm.at[idx]=='':r.append('MISSING_SHIPMENT_ID')
    if failures.at[idx,'ShipDate']:r.append('INVALID_MARCH_DATE')
    if pd.isna(weights.at[idx]):r.append('NONNUMERIC_WEIGHT')
    elif weights.at[idx]<=0:r.append('NONPOSITIVE_WEIGHT')
    if pd.isna(costs.at[idx]):r.append('NONNUMERIC_COST')
    if failures.at[idx,'CustomerID']:r.append('UNRESOLVED_CUSTOMER')
    reasons.append(';'.join(r))
reasons=pd.Series(reasons,index=staging.index)
reject_mask=reasons.ne('')
reject_log=pd.DataFrame({
 'BatchID':RUN_ID,'SourceFile':staging.loc[reject_mask,'SourceFile'],
 'SourceRow':staging.loc[reject_mask,'SourceRow'],
 'RawRecord':staging.loc[reject_mask,'RawRecord'],
 'ReasonCode':reasons[reject_mask],'RejectedAtUTC':AUDIT_TIME})
reject_log.to_csv(OUT/'RejectLog.csv',index=False)
loadable=staging.loc[~reject_mask].copy()
print('Reject classes:',reject_log.ReasonCode.value_counts().to_dict())

Reject classes: {'NONNUMERIC_WEIGHT': 9, 'INVALID_MARCH_DATE': 6, 'NONPOSITIVE_WEIGHT': 5, 'MISSING_SHIPMENT_ID': 4}


## Step 05 - Conform types and source-specific units; canonical consignment ID

In [6]:
clean=loadable.copy()
clean['ShipmentID']=id_norm.loc[clean.index]
rail=clean.SourceMode.eq('Rail')
clean.loc[rail,'ShipmentID']=clean.loc[rail,'ShipmentID'].map(lambda v:v if v.startswith('RL-') else 'RL-'+v)
clean['ShipDate']=parsed_dates.loc[clean.index]
clean['WeightKG']=weights.loc[clean.index]*np.where(rail,1000,1)
clean['FreightCostPKR']=costs.loc[clean.index]
clean['OriginCityKey']=origin.loc[clean.index].fillna(-1).astype(int)
clean['DestCityKey']=destination.loc[clean.index].fillna(-1).astype(int)
clean['StatusKey']=status.loc[clean.index].fillna(-1).astype(int)

## Step 06 - Deduplicate at one row per canonical consignment

In [7]:
raw_exact=len(loadable.drop_duplicates(CONTRACT))
raw_id=len(loadable.drop_duplicates('ShipmentID'))
dup_mask=clean.ShipmentID.duplicated(keep='first')
removed=clean.loc[dup_mask].copy()
keepers=clean.loc[~dup_mask].copy()
first_source=keepers.set_index('ShipmentID')[['SourceFile','SourceRow']].to_dict('index')
removed['RetainedSource']=removed.ShipmentID.map(lambda k:json.dumps(first_source[k]))
removed['Reason']='DUPLICATE_CANONICAL_SHIPMENT_ID'
removed.to_csv(OUT/'DuplicateLog.csv',index=False)

## Step 07 - Conform customer businesses and preserve all registry aliases

In [8]:
# Match spelling variants using an explicit, reproducible business-name rule.
# This is the six known alias pairs in this exercise, not a universal identity algorithm.
def business_name(v):
    t=norm(v)
    t=re.sub(r'\(PVT\)',' ',t)
    t=re.sub(r'[^A-Z0-9 ]',' ',t)
    t=re.sub(r'\b(?:LIMITED|LTD)\b',' ',t)
    t=re.sub(r'\bCO\b','COMPANY',t)
    return ' '.join(t.split())
registry['BusinessName']=registry.CustomerName.map(business_name)
registry['CustomerKey']=pd.factorize(registry.BusinessName,sort=False)[0]+1
customer_aliases=registry.copy()
customer_aliases.to_csv(OUT/'CustomerAliases.csv',index=False)
rows=[]
for key,g in registry.groupby('CustomerKey',sort=True):
    first=g.iloc[0]
    rows.append({'CustomerKey':key,'CustomerName':first.CustomerName,'City':first.City,'Sector':first.Sector,
      'SignedUp':g.SignedUp.min(),'CustomerIDAliases':' | '.join(g.CustomerID),
      'NameAliases':' | '.join(g.CustomerName)})
dim_customer=pd.DataFrame([{'CustomerKey':-1,'CustomerName':'Unknown','City':'Unknown','Sector':'Unknown',
 'SignedUp':'','CustomerIDAliases':'','NameAliases':''}]+rows)
customer_lookup=dict(zip(registry.CustomerID,registry.CustomerKey))
keepers['CustomerKey']=keepers.CustomerID.map(trim).map(customer_lookup).fillna(-1).astype(int)

## Step 08 - Build gapless quarter date dimension and resolve all foreign keys

In [9]:
# The required 90 rows imply 2026 Q1, not only March's 31 days.
dates=pd.date_range('2026-01-01','2026-03-31')
dim_date=pd.DataFrame({'DateKey':dates.strftime('%Y%m%d').astype(int),'FullDate':dates.strftime('%Y-%m-%d'),
 'Year':dates.year,'Quarter':dates.quarter,'MonthNumber':dates.month,'MonthName':dates.month_name(),
 'DayOfMonth':dates.day,'DayOfWeek':(dates.dayofweek+1)%7+1,'DayName':dates.day_name(),
 'IsWeekend':np.where(dates.dayofweek>=5,'Yes','No'),
 'IsHoliday':'Unknown'})
# No holiday calendar was supplied; do not invent public holiday flags.
keepers['DateKey']=keepers.ShipDate.dt.strftime('%Y%m%d').astype(int)
keepers['ModeKey']=keepers.SourceMode.map(dict(zip(dim_mode.ModeName,dim_mode.ModeKey)))
route_lookup={(r.OriginCityKey,r.DestCityKey):r.RouteKey for r in dim_route.itertuples()}
keepers['RouteKey']=[route_lookup.get((o,d),-1) for o,d in zip(keepers.OriginCityKey,keepers.DestCityKey)]
keepers['ShipmentCount']=1
# Eight identifiers (seven dimension FKs + consignment ID), three additive facts.
FACT_COLS=['ShipmentID','DateKey','OriginCityKey','DestCityKey','RouteKey','ModeKey','StatusKey','CustomerKey',
 'WeightKG','FreightCostPKR','ShipmentCount']
fact=keepers[FACT_COLS].reset_index(drop=True)
unknown_rows=keepers[(keepers.OriginCityKey==-1)|(keepers.DestCityKey==-1)|(keepers.StatusKey==-1)].copy()
unknown_rows[['ShipmentID','SourceFile','SourceRow','Origin','Destination','StatusRaw','OriginCityKey','DestCityKey','StatusKey']].to_csv(OUT/'UnknownLookupAudit.csv',index=False)
for name,df in [('FactShipment_March',fact),('DimCustomer',dim_customer),('DimDate',dim_date),
 ('DimCity',dim_city),('DimRoute',dim_route),('DimMode',dim_mode),('DimStatus',dim_status)]:
    df.to_csv(OUT/f'{name}.csv',index=False)

## Step 09 - Reconciliation, ten definition-of-done assertions and referential checks

In [10]:
actual=[len(staging),len(reject_log),len(removed),len(fact),fact.WeightKG.sum(),fact.FreightCostPKR.sum(),
 int(((fact.OriginCityKey==-1)|(fact.DestCityKey==-1)).sum()),int((fact.StatusKey==-1).sum()),
 len(dim_customer[dim_customer.CustomerKey!=-1]),len(dim_date)]
expected=[1284,24,60,1200,17542050,147037000,14,7,40,90]
labels=['Extracted data rows','Rejected','Deduplicated','Fact rows','WeightKG','FreightCostPKR',
 'Unknown city rows','Unknown status rows','Known customers','Calendar days']
checks=pd.DataFrame({'Check':labels,'Actual':actual,'Expected':expected})
checks['Pass']=checks.Actual.eq(checks.Expected)
checks.to_csv(OUT/'T7_Checks.csv',index=False)
for row in checks.itertuples():assert row.Pass, f'{row.Check}: {row.Actual} != {row.Expected}'
assert len(staging)-len(fact)-len(reject_log)-len(removed)==0
assert fact.ShipmentID.is_unique and fact.notna().all().all()
assert (fact.FreightCostPKR<0).sum()==3
for fk,dim,pk in [('DateKey',dim_date,'DateKey'),('OriginCityKey',dim_city,'CityKey'),
 ('DestCityKey',dim_city,'CityKey'),('RouteKey',dim_route,'RouteKey'),('ModeKey',dim_mode,'ModeKey'),
 ('StatusKey',dim_status,'StatusKey'),('CustomerKey',dim_customer,'CustomerKey')]:
    assert dim[pk].is_unique and fact[fk].isin(dim[pk]).all(),fk
print(f'Extracted {len(staging):,} = loaded {len(fact):,} + rejected {len(reject_log)} + deduplicated {len(removed)}')
print(checks.to_string(index=False))

Extracted 1,284 = loaded 1,200 + rejected 24 + deduplicated 60
              Check      Actual  Expected  Pass
Extracted data rows      1284.0      1284  True
           Rejected        24.0        24  True
       Deduplicated        60.0        60  True
          Fact rows      1200.0      1200  True
           WeightKG  17542050.0  17542050  True
     FreightCostPKR 147037000.0 147037000  True
  Unknown city rows        14.0        14  True
Unknown status rows         7.0         7  True
    Known customers        40.0        40  True
      Calendar days        90.0        90  True


## Step 10 - Measured evidence for T1, T4, T5 and T6

In [11]:
natural=['ShipDate','OriginCityKey','DestCityKey','WeightKG','CustomerKey']
collisions=keepers[keepers.duplicated(natural,keep=False)].sort_values(natural)
collisions[['ShipmentID','ShipDate','OriginCityKey','DestCityKey','WeightKG','CustomerID','CustomerKey']].to_csv(OUT/'NaturalKeyCollisions.csv',index=False)
counts=[raw_exact,raw_id,len(keepers),len(keepers.drop_duplicates(natural))]
duplicate_defs=pd.DataFrame({'Definition':['Whole raw contract row','Raw ShipmentID','Canonical ShipmentID','Day + lane + weight + customer'], 'RowsSurviving':counts})
duplicate_defs.to_csv(OUT/'T5_DuplicateDefinitions.csv',index=False)
idx_cols=FACT_COLS[:8]
selectivity=pd.DataFrame([{'Column':c,'Distinct':fact[c].nunique(),'Selectivity':fact[c].nunique()/len(fact),
 'MeanRowsPerValue':len(fact)/fact[c].nunique(),'WorstValueRows':fact[c].value_counts().max()} for c in idx_cols])
selectivity.to_csv(OUT/'T6_Selectivity.csv',index=False)
ladder=[]
for filename,mode,_ in SPECS:
    d=staging[staging.SourceFile==filename]
    ladder.append({'Stage':filename,'Rows':len(d),'NumericWeightAsIssued':d.WeightKG.map(decimal_number).sum(),
       'FreightCostPKR':d.FreightCostPKR.map(decimal_number).sum(),'WeightUnit':'tonnes' if mode=='Rail' else 'kg; invalid text skipped'})
ladder.extend([{'Stage':'Staging','Rows':len(staging)},{'Stage':'Rejected','Rows':len(reject_log)},
 {'Stage':'Deduplicated','Rows':len(removed)}, {'Stage':'Loaded','Rows':len(fact),'NumericWeightAsIssued':fact.WeightKG.sum(),'FreightCostPKR':fact.FreightCostPKR.sum(),'WeightUnit':'kg'}])
pd.DataFrame(ladder).to_csv(OUT/'T4_Ladder.csv',index=False)
unknown_city=keepers.loc[keepers.OriginCityKey==-1,'Origin'].value_counts().to_dict()
unknown_status=keepers.loc[keepers.StatusKey==-1,'StatusRaw'].value_counts().to_dict()
weighted=fact.FreightCostPKR.sum()/fact.WeightKG.sum()
unweighted=(fact.FreightCostPKR/fact.WeightKG).mean()
evidence={'actual':actual,'duplicate_counts':counts,'unknown_city':unknown_city,'unknown_status':unknown_status,
 'weighted_rate':weighted,'mean_row_rate':unweighted,'negative_weight_rows':int(weights.lt(0).sum()),
 'negative_cost_rows':int(costs.lt(0).sum()),'collision_ids':collisions.ShipmentID.tolist(),
 'mode_counts':fact.ModeKey.value_counts().to_dict(), 'stale_dates':len(stale_date),
 'raw_customer_ids':keepers.CustomerID.nunique(),'natural_survivors':counts[-1]}
(OUT/'Evidence.json').write_text(json.dumps(evidence,indent=2,default=lambda v:int(v)),encoding='utf-8')
print('Duplicate definitions:',counts,'\nNatural-key collisions:',collisions.ShipmentID.tolist())
print('Unresolved cities:',unknown_city,'\nUnresolved statuses:',unknown_status)
print('Cost/kg: weighted',round(weighted,6),'versus mean row ratio',round(unweighted,6))

Duplicate definitions: [1212, 1212, 1200, 1198] 
Natural-key collisions: ['RD-5216', 'RD-5337', 'RD-5409', 'RD-5332']
Unresolved cities: {'Keti Bandar Jetty': 3, 'Kotri Junction': 3, 'Wagah Border': 2, 'Taftan Crossing': 2, 'Torkham Gate': 2, 'Chaman Border': 2} 
Unresolved statuses: {'held at customs': 4, 'RTO': 2, 'Hold': 1}
Cost/kg: weighted 8.381974 versus mean row ratio 7.745634


## Step 11 - Demonstrate a DateKey index and province join with SQLite

In [12]:
# SQLite is a reproducible SQL illustration, not a Power BI performance benchmark.
with sqlite3.connect(OUT/'PakFreight_March.sqlite') as conn:
    for name,df in [('FactShipment_March',fact),('DimCity',dim_city),('DimCustomer',dim_customer),('DimDate',dim_date)]:
        df.to_sql(name,conn,index=False,if_exists='replace')
    sql='SELECT SUM(WeightKG) FROM FactShipment_March WHERE DateKey BETWEEN 20260301 AND 20260307'
    before=conn.execute('EXPLAIN QUERY PLAN '+sql).fetchall()
    conn.execute('CREATE INDEX IF NOT EXISTS IX_FactShipment_DateKey ON FactShipment_March(DateKey)')
    after=conn.execute('EXPLAIN QUERY PLAN '+sql).fetchall()
    province=pd.read_sql_query('SELECT d.Province, SUM(f.WeightKG) AS WeightKG FROM FactShipment_March f JOIN DimCity d ON f.OriginCityKey=d.CityKey GROUP BY d.Province',conn)
    province.to_csv(OUT/'WeightByOriginProvince.csv',index=False)
    plan={'BeforeIndex':before,'AfterIndex':after,'NewIndexEntriesForMarch':len(fact)}
    (OUT/'IndexPlan.json').write_text(json.dumps(plan,indent=2))
print('Before index:',before,'\nAfter index:',after)
print('ALL TEN CHECKS PASS')

Before index: [(3, 0, 216, 'SCAN FactShipment_March')] 
After index: [(4, 0, 163, 'SEARCH FactShipment_March USING INDEX IX_FactShipment_DateKey (DateKey>? AND DateKey<?)')]
ALL TEN CHECKS PASS


## T1 — Carry-forward audit

Week labels map the claim topics to the weeks named in the brief.

|   Claim | Week (topic mapping)   | Verdict   | Numeric evidence                                 | Explanation                                                                  |
|--------:|:-----------------------|:----------|:-------------------------------------------------|:-----------------------------------------------------------------------------|
|       1 | W2                     | True      | 1,284 data rows; 1,200 consignments              | 84 surplus data rows: 24 rejects and 60 duplicate copies.                    |
|       2 | W1                     | False     | 5 negative weights; 3 negative costs             | The 3 negative costs are real credit notes and must load.                    |
|       3 | W2                     | False     | 4,683,000 kg; 9,218.25 t; 4,487,200 kg           | Raw SUM skips text and mixes units; duplicates and rejects remain.           |
|       4 | W2                     | False     | 1,286 physical = 1,284 data + 2 structural       | Land the footer first, then classify it; keep StructuralAudit.               |
|       5 | W4                     | True      | 1,200 rows; 1,200 canonical IDs                  | Exactly one row per consignment; the key is unique.                          |
|       6 | W1                     | False     | 8.381974 vs 7.745634 PKR/kg                      | Use total cost / total weight; a mean of ratios weights small loads equally. |
|       7 | W3                     | False     | 4 crossings, 8 rows; 0 additions                 | They are facilities without an approved city mapping; use -1.                |
|       8 | W4                     | False     | 28 February dates; 0 March keys                  | Rebuild 90 gapless Q1 dates; old DimDate has no March members.               |
|       9 | W4                     | False     | 1,200 vs 3,600 probes                            | The three-join snowflake traverses 3 times as many joins.                    |
|      10 | W4                     | True      | 1,200 facts to 15 city members                   | Each city FK has one dimension match; filter dimension to fact.              |
|      11 | W5                     | False     | 1,212 before trim; 1,200 after                   | The 12 trailing-space copies survive early deduplication.                    |
|      12 | W6                     | False     | 1,200 rows; exact totals; 0 valid March date FKs | Keeping February DimDate preserves both sums but breaks every date join.     |
|      13 | W6                     | False     | 6 rejected dates; 14 city -1 rows                | Dates define valid fact grain; cities can use a known Unknown member.        |
|      14 | W6                     | False     | 3/1,200 = 0.0025; road = 660 rows                | Mode has poor discrimination. Date has 31 values and useful ranges.          |
|      15 | W5                     | False     | 1,198 survivors; 2 real shipments lost           | RD-5216 / RD-5337 and RD-5332 / RD-5409 share that business key.             |

Exactly **3 True**: claims 1, 5 and 10.

## T2 — Profile before transforms

| Column         |   Blanks |   DistinctRaw |   RuleFailures | NeedsTextTransform   |
|:---------------|---------:|--------------:|---------------:|:---------------------|
| ShipmentID     |        4 |          1232 |             64 | 712                  |
| ShipDate       |        0 |            98 |              6 | —                    |
| Origin         |        0 |            18 |             14 | 570                  |
| Destination    |        0 |            19 |              0 | 189                  |
| WeightKG       |        0 |            78 |             14 | —                    |
| FreightCostPKR |        0 |           378 |              0 | —                    |
| StatusRaw      |        0 |            25 |              7 | 980                  |
| CustomerID     |        0 |            46 |              0 | —                    |

**Failure total = 105**. ShipmentID counts 4 blanks plus 60 surplus copies, not every member of each duplicate group. Weight has 9 text exceptions plus 5 negatives. Text-transform counts compare valid raw values to canonical display spellings; numeric rail IDs also need an RL- prefix during source conformance.

Profiling raw values preserves the evidence of what arrived. Trimming, casing, unit conversion and deduplication hide source defects and reduce the counts. The original profile records 105 column-rule failures, although only 24 rows must be rejected. Keep source lineage and raw values so a corrected source can be tested against the same rules.

CustomerID is clean at the lookup level: 0 blanks and 0 unmatched IDs; however, the registry has 46 IDs for 40 businesses. StatusRaw has 25 raw strings but only 3 supported statuses plus 3 unresolved labels after casing/trim, so raw variety overstates its defects. Also, 98 raw dates contain 3 source representations, and all 378 distinct cost values are valid. Distinctness measures variety; failure counts measure broken rules.

Quality risks: ID completeness/uniqueness; date validity; city consistency/completeness; weight validity; cost validity; status consistency; customer identity consistency.

## T3 — Reject contract

| Class               |   Rows | Verdict             | Reason / consequence                                                                                 |
|:--------------------|-------:|:--------------------|:-----------------------------------------------------------------------------------------------------|
| Text weight         |      9 | Reject              | 9 non-contract weights need source confirmation; guessing numeric values changes business data.      |
| Invalid March date  |      6 | Reject              | 6 dates cannot identify a valid March day; trimming or a locale change cannot repair them.           |
| Negative weight     |      5 | Reject              | 5 negative weights violate positive physical shipment weight; do not confuse them with credit notes. |
| Blank ShipmentID    |      4 | Reject              | 4 blank IDs cannot identify a consignment or support safe deduplication.                             |
| Negative cost       |      3 | Load                | 3 credit notes are valid negative costs; rejecting them overstates net freight revenue.              |
| Unresolved origin   |     14 | Route to Unknown    | 14 real shipments keep their facts and use city -1; deleting them loses 195,150 kg.                  |
| Unresolved status   |      7 | Route to Unknown    | 7 real shipments keep their facts and use status -1; deleting them hides business activity.          |
| Exact copied row    |     48 | Remove as duplicate | 48 exact copies add no shipment; keeping them double counts the facts.                               |
| Trailing-space copy |     12 | Remove as duplicate | 12 trailing-space copies match after trim/upper; keeping them adds 171,900 kg.                       |

Class totals: **24 Reject; 21 Unknown; 60 duplicate copies removed; 3 credit notes loaded**. These nine exception classes do not enumerate all normal shipments.

### Six-column RejectLog

| Column        | Type          | Purpose                                                                              |
|:--------------|:--------------|:-------------------------------------------------------------------------------------|
| BatchID       | text          | Identifies the source batch and pipeline run for traceable reprocessing.             |
| SourceFile    | text          | Exact input filename; selects the source and its schema.                             |
| SourceRow     | integer       | Physical CSV record line or XLSX worksheet row; points to the source location.       |
| RawRecord     | JSON text     | All original source fields and values, including the original ID and rejected value. |
| ReasonCode    | text          | All failed load rules; tells the source owner what must be corrected.                |
| RejectedAtUTC | UTC timestamp | Time of the routing decision; supports audit and correction history.                 |

Reject when an essential consignment attribute is invalid: the ID is absent, the date is unusable, or weight is invalid. A loadable shipment with an unresolved descriptor still has valid measures and loads on -1. Routing invalid weight/date to Unknown makes invalid facts look usable; rejecting an unresolved city discards actual freight. RejectLog keeps the original row for correction, while UnknownLookupAudit exposes unresolved attributes for later remapping.

**Resubmission:** locate the original file and physical row; read RawRecord; correct the fields identified by ReasonCode in the source system; submit a new batch through the same validations and canonical-ID deduplication. Keep the old reject audit unchanged.

## T4 — Reconciliation

| Stage            |   Rows | NumericWeightAsIssued   | FreightCostPKR   | WeightUnit               |
|:-----------------|-------:|:------------------------|:-----------------|:-------------------------|
| marine_march.csv |    261 | 4683000.0               | 35571000.0       | kg; invalid text skipped |
| rail_march.xlsx  |    315 | 9218.25                 | 88702500.0       | tonnes                   |
| road_march.csv   |    708 | 4487200.0               | 32259500.0       | kg; invalid text skipped |
| Staging          |   1284 | —                       | —                | —                        |
| Rejected         |     24 | —                       | —                | —                        |
| Deduplicated     |     60 | —                       | —                | —                        |
| Loaded           |   1200 | 17542050.0              | 147037000.0      | kg                       |

The raw source weight subtotals skip invalid text; the rail subtotal is tonnes. They are not additive kg controls. The road footer repeats 4,487,200 and 32,259,500 and is excluded from the data subtotal but retained in StructuralAudit.

**1,284 − 1,200 − 24 − 60 = 0.** Physical landing additionally reconciles as **1,286 = 1,284 + 2 structural records**.

| Given result            | Cause and checks                                                                                                                                                   |
|:------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 1,212 / 17,713,950 kg   | Deduplicated before trim: 12 space variants remain (+171,900 kg). T7 checks 3, 4 and 5 fail.                                                                       |
| 1,200 / 8,603,247.75 kg | Rail tonnes loaded as kg: 8,947,750 kg became 8,947.75; short by 8,938,802.25 kg. T7 check 5 fails.                                                                |
| 1,209 / 17,542,050 kg   | 9 invalid-text weights loaded as null: SUM skips them, so weight passes. T7 checks 2 and 4 fail. Their source costs add PKR 1,081,500, so cost also fails if kept. |
| 1,186 / 17,346,900 kg   | Deleted the 14 unresolved-city shipments: lost 195,150 kg. T7 checks 4, 5 and 7 fail.                                                                              |

**Correction to the brief:** keeping the nine rejected text-weight rows with null weight preserves the weight SUM but adds **PKR 1,081,500** if their original costs survive. The claim that cost is also exact requires another undocumented change to those costs. The source files do not support that claim as written.

## T5 — Duplicates and customer identity

| Definition                               |   Surviving rows |
|:-----------------------------------------|-----------------:|
| Whole raw contract row                   |             1212 |
| Raw ShipmentID                           |             1212 |
| Trimmed, upper, prefixed ShipmentID      |             1200 |
| Day + lane + weight + conformed customer |             1198 |

Raw row comparison excludes audit metadata such as SourceRow. Otherwise no copied record would be identical. The two destroyed consignment pairs are shown below.

| ShipmentID   | ShipDate   |   OriginCityKey |   DestCityKey |   WeightKG | CustomerID   |   CustomerKey |
|:-------------|:-----------|----------------:|--------------:|-----------:|:-------------|--------------:|
| RD-5216      | 2026-03-15 |               3 |             9 |       6300 | CU-136       |            36 |
| RD-5337      | 2026-03-15 |               3 |             9 |       6300 | CU-136       |            36 |
| RD-5409      | 2026-03-29 |               1 |            11 |       6500 | CU-102       |             2 |
| RD-5332      | 2026-03-29 |               1 |            11 |       6500 | CU-102       |             2 |

Registry rows **46**; distinct source CustomerIDs used in March **46**; conformed served businesses **40**; DimCustomer rows including Unknown **41**. The final fact uses CustomerKey, while CustomerAliases and lineage retain the original IDs.

Six businesses each have two registry IDs under spelling variants. Conform names, assign one stable CustomerKey per business, and preserve all 46 IDs and spellings in CustomerAliases. The raw distinct-ID count is 46; the served-business count is 40. Remapping customer FKs changes grouping only: no fact row or measure changes, so weight and cost sums stay identical. The dimension has 41 rows including Unknown.

## T6 — Index and join path

| Column        |   Distinct |   Selectivity |   MeanRowsPerValue |   WorstValueRows |
|:--------------|-----------:|--------------:|-------------------:|-----------------:|
| ShipmentID    |       1200 |      1.000000 |           1.000000 |                1 |
| DateKey       |         31 |      0.025833 |          38.709677 |               40 |
| OriginCityKey |          9 |      0.007500 |         133.333333 |              605 |
| DestCityKey   |         13 |      0.010833 |          92.307692 |              175 |
| RouteKey      |         21 |      0.017500 |          57.142857 |              113 |
| ModeKey       |          3 |      0.002500 |         400.000000 |              660 |
| StatusKey     |          4 |      0.003333 |         300.000000 |              821 |
| CustomerKey   |         40 |      0.033333 |          30.000000 |               57 |

Build a DateKey index for the date ranges used on every page and for append-by-date locality. It has 31 values (0.025833) and at most 40 rows per date. ShipmentID has 1.0 distinctness but no dashboard filter demand. ModeKey has 0.0025 and road covers 660/1,200 rows, so a scan often wins. One added index requires 1,200 index-entry insertions for this full March load, one per new fact row, plus storage and maintenance; a second CustomerKey index adds another 1,200. SQLite EXPLAIN changes from SCAN to indexed SEARCH. Before tuning: remove descriptive text from facts, use single-direction relationships, and calculate SUM(cost)/SUM(weight) once.

| Schema | Tables | Joins | Simplified probes |
|---|---:|---:|---:|
| Fact → DimCity | 2 | 1 | 1,200 |
| Fact → DimCity → DimProvince → DimCountry | 4 | 3 | 3,600 |

The exercise defines probes as rows × joins; real query-plan costs depend on algorithm and aggregation placement. A province-only question need not traverse Country in an optimized snowflake, but the requested path includes it.

**Terminology correction:** distinct/rows is an index discrimination statistic, not the fraction of rows read by a lookup. The average equality-match fraction is approximately 1/distinct under uniformity, and actual skew matters. The supplied table actually has Marine 240, Rail 300, Road 660 rows, not 400 each. Date index write cost is **1,200 added entries** for this load, not a claim of 1,200 physical disk writes.

## T7 — Definition of done

| Check               |         Actual |   Expected | Pass   |
|:--------------------|---------------:|-----------:|:-------|
| Extracted data rows | 1284           |       1284 | True   |
| Rejected            |   24           |         24 | True   |
| Deduplicated        |   60           |         60 | True   |
| Fact rows           | 1200           |       1200 | True   |
| WeightKG            |    1.7542e+07  |   17542050 | True   |
| FreightCostPKR      |    1.47037e+08 |  147037000 | True   |
| Unknown city rows   |   14           |         14 | True   |
| Unknown status rows |    7           |          7 | True   |
| Known customers     |   40           |         40 | True   |
| Calendar days       |   90           |         90 | True   |

**10 / 10 pass**. Additional assertions verify every dimension FK, unique fact grain, non-null facts, the 3 credit notes and the full physical-record identity.

I counted unmatched lookups after deduplication and exported their file, row and raw labels. Origin city -1 = 14: Keti Bandar Jetty 3, Kotri Junction 3, Wagah Border 2, Taftan Crossing 2, Torkham Gate 2, Chaman Border 2. Status -1 = 7: held at customs 4, RTO 2, Hold 1. These are disjoint sets: 21 shipments are routed. Deleting the 14 city rows would make that defect count zero while hiding 195,150 kg and reducing the fact to 1,186 rows. Unknown preserves measures and makes missing mappings visible.

The complete UnknownLookupAudit lists every affected consignment. The three credit notes are MR-2203 (−195,000), RD-5568 (−50,000) and RD-5224 (−38,000 PKR). Their total is −283,000 PKR.

In [13]:
# Final review of exported outputs
print('Notebook complete: ten definition-of-done checks passed.')
print('Fact:', fact.shape, 'RejectLog:', reject_log.shape)
display(checks)
display(pd.DataFrame(sorted(p.name for p in OUT.iterdir()), columns=['Output file']))

Notebook complete: ten definition-of-done checks passed.
Fact: (1200, 11) RejectLog: (24, 6)
                 Check       Actual   Expected  Pass
0  Extracted data rows       1284.0       1284  True
1             Rejected         24.0         24  True
2         Deduplicated         60.0         60  True
3            Fact rows       1200.0       1200  True
4             WeightKG   17542050.0   17542050  True
5       FreightCostPKR  147037000.0  147037000  True
6    Unknown city rows         14.0         14  True
7  Unknown status rows          7.0          7  True
8      Known customers         40.0         40  True
9        Calendar days         90.0         90  True
                    Output file
0           CustomerAliases.csv
1                   DimCity.csv
2               DimCustomer.csv
3                   DimDate.csv
4                   DimMode.csv
5                  DimRoute.csv
6                 DimStatus.csv
7              DuplicateLog.csv
8                 Evidence.json
9   